In [0]:
%sql
CREATE CATALOG IF NOT EXISTS atliq
MANAGED LOCATION 'abfss://lakehouse@atliqcommerce.dfs.core.windows.net/atliq-managed';
CREATE SCHEMA  IF NOT EXISTS atliq.silver;
CREATE SCHEMA  IF NOT EXISTS atliq.gold;

In [0]:
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable


In [0]:
BRONZE = "abfss://lakehouse@atliqcommerce.dfs.core.windows.net/bronze"

In [0]:
bronze_customers_df = spark.read.parquet(
    f"{BRONZE}/customers"
)

silver_customers_df = (
    bronze_customers_df
    .withColumn("city", F.initcap(F.trim("city")))
    .withColumn("signup_date", F.to_date("signup_date"))
    .dropDuplicates(["customer_id"])
    .filter(F.col("customer_id").isNotNull())
)

(
    silver_customers_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("atliq.silver.customers")
)

In [0]:
BRONZE = "abfss://lakehouse@atliqcommerce.dfs.core.windows.net/bronze"

bronze_products_df = spark.read.parquet(
    f"{BRONZE}/products"
)

silver_products_df = (
    bronze_products_df
    .withColumn("product_name", F.trim(F.col("product_name")))
    .withColumn("category", F.trim(F.col("category")))
    .filter(F.col("product_id").isNotNull())
    .dropDuplicates(["product_id"])
)

(
    silver_products_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("atliq.silver.products")
)

In [0]:
bronze_marketing_spend_df = spark.read.parquet(
    f"{BRONZE}/marketing_spend"
)

silver_marketing_spend_df = (
    bronze_marketing_spend_df
    .withColumn("spend_date", F.to_date(F.col("spend_date")))
    .withColumn(
        "spend_amount",
        F.col("spend_amount").cast("decimal(12,2)")
    )
    .withColumn("channel", F.trim(F.col("channel")))
    .withColumn("campaign", F.trim(F.col("campaign")))
    .filter(F.col("spend_date").isNotNull())
    .filter(F.col("spend_amount").isNotNull())
    .filter(F.col("clicks").isNotNull())
)

(
    silver_marketing_spend_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("atliq.silver.marketing_spend")
)

In [0]:
bronze_supplier_price_list_df = spark.read.parquet(
    f"{BRONZE}/supplier_price_list"
)

silver_supplier_price_list_df = (
    bronze_supplier_price_list_df
    .withColumn("effective_date", F.to_date(F.col("effective_date")))
    .withColumn(
        "supplier_cost",
        F.col("supplier_cost").cast("decimal(12,2)")
    )
    .withColumn("product_name", F.trim(F.col("product_name")))
    .withColumn("supplier_name", F.trim(F.col("supplier_name")))
    .filter(F.col("product_id").isNotNull())
    .filter(F.col("supplier_cost").isNotNull())
)

(
    silver_supplier_price_list_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("atliq.silver.supplier_price_list")
)

In [0]:
run_date = dbutils.widgets.get("run_date")

src_path = f"{BRONZE}/orders/ingest_date={run_date}"

batch_df = spark.read.parquet(src_path)

# Keep the latest version of each order in this batch
window_spec = (
    Window
    .partitionBy("order_id")
    .orderBy(F.col("updated_at").desc())
)

orders_df = (
    batch_df
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .withColumn("order_date", F.to_date("order_date"))
    .withColumn(
        "order_amount",
        F.col("order_amount").cast("decimal(12,2)")
    )
    .filter(F.col("order_id").isNotNull())
)

# Create the target table if it does not exist
(
    DeltaTable
    .createIfNotExists(spark)
    .tableName("atliq.silver.orders")
    .addColumns(orders_df.schema)
    .execute()
)

# Upsert the batch into Silver
(
    DeltaTable
    .forName(spark, "atliq.silver.orders")
    .alias("target")
    .merge(
        orders_df.alias("source"),
        "target.order_id = source.order_id"
    )
    .whenMatchedUpdateAll(
        condition="source.updated_at > target.updated_at"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
run_date = dbutils.widgets.get("run_date")

src_path = f"{BRONZE}/order_items/ingest_date={run_date}"

batch_df = spark.read.parquet(src_path)

# Deduplicate the current batch by order_item_id
silver_order_items_df = (
    batch_df
    .dropDuplicates(["order_item_id"])
    .filter(F.col("order_item_id").isNotNull())
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("product_id").isNotNull())
    .filter(F.col("quantity") > 0)
    .filter(F.col("item_price") >= 0)
)

# Create Silver Delta table on the first run
(
    DeltaTable
    .createIfNotExists(spark)
    .tableName("atliq.silver.order_items")
    .addColumns(silver_order_items_df.schema)
    .execute()
)

# Insert only new order items
(
    DeltaTable
    .forName(spark, "atliq.silver.order_items")
    .alias("target")
    .merge(
        silver_order_items_df.alias("source"),
        "target.order_item_id = source.order_item_id"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
run_date = dbutils.widgets.get("run_date")

src_path = f"{BRONZE}/payments/ingest_date={run_date}"

batch_df = spark.read.parquet(src_path)

# Keep the latest version of each payment in this batch
window_spec = (
    Window
    .partitionBy("payment_id")
    .orderBy(F.col("updated_at").desc())
)

silver_payments_df = (
    batch_df
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .withColumn("amount", F.col("amount").cast("decimal(12,2)"))
    .withColumn("method", F.trim(F.col("method")))
    .withColumn("paid_at", F.to_timestamp(F.col("paid_at")))
    .withColumn("updated_at", F.to_timestamp(F.col("updated_at")))
    .filter(F.col("payment_id").isNotNull())
    .filter(F.col("order_id").isNotNull())
)

(
    DeltaTable
    .createIfNotExists(spark)
    .tableName("atliq.silver.payments")
    .addColumns(silver_payments_df.schema)
    .execute()
)

(
    DeltaTable
    .forName(spark, "atliq.silver.payments")
    .alias("target")
    .merge(
        silver_payments_df.alias("source"),
        "target.payment_id = source.payment_id"
    )
    .whenMatchedUpdateAll(
        condition="source.updated_at > target.updated_at"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
BRONZE = "abfss://lakehouse@atliqcommerce.dfs.core.windows.net/gold"
bronze_products_df = spark.read.format('delta').load(
    f"{BRONZE}/dim_products")

bronze_products_df.count()

